# Training pipeline
Setup training variables in `training/config.yaml`. Download and store the transaction table and it's metadata under `base_data_dir` as defined in the config.

Make sure the following files exist under the data directory (e.g. `single_table/data/`):

| File | Role |
|------|------|
| `trans.csv` | Train split used to fit the diffusion model |
| `trans_holdout.csv` | Holdout split reserved for evaluation (not used in training) |
| `trans_domain.json` | Per-column `type` (`continuous` / `discrete`) |
| `dataset_meta.json` | Table graph: tables and parent/child relations (single-table: one table, no parents) |
| `meta_info.json` | Evaluation schema: numeric/categorical column indices, target column, and task type (e.g. `regression`, `binary_classification`) |


In [6]:
import os
from pathlib import Path


# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    """Change the working directory to the repository root."""
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


set_project_root()

PosixPath('/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp')

In [ ]:
# Source: https://github.com/VectorInstitute/midst-toolkit/blob/main/examples/training/single_table/run_training.py
import pickle
from logging import INFO

from hydra import compose, initialize
from midst_toolkit.common.config import ClavaDDPMDiffusionConfig
from midst_toolkit.common.logger import TOOLKIT_LOGGER, log
from midst_toolkit.common.variables import DEVICE
from midst_toolkit.models.clavaddpm.data_loaders import load_tables
from midst_toolkit.models.clavaddpm.train import ClavaDDPMModelArtifacts, clava_training
from midst_toolkit.common.random import set_all_random_seeds
from omegaconf import OmegaConf


# Preventing some excessive logging
TOOLKIT_LOGGER.setLevel(INFO)

/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [7]:
ROOT = Path.cwd()
REFERENEC_ROOT = ROOT / "implementations" / "tabular_data" / "single_table"
# Set data and output directories
base_data_dir = REFERENEC_ROOT / "data"
base_output_dir = REFERENEC_ROOT / "results"

## Load and initialize hydra config

In [ ]:
# Context manager ensures global state is cleaned up after initialization
with initialize(version_base=None, config_path="."):
    # Load config.yaml and pass optional command-line style overrides
    cfg = compose(config_name="config")

# View the configuration as a standard YAML string
print(OmegaConf.to_yaml(cfg))
# Load and set the random seeds
if cfg.random_seed is not None:
    set_all_random_seeds(seed=cfg.random_seed)

diffusion_config:
  d_layers:
  - 512
  - 1024
  - 1024
  - 1024
  - 1024
  - 512
  dropout: 0.0
  num_timesteps: 2000
  model_type: mlp
  iterations: 20000
  batch_size: 4096
  lr: 0.0006
  gaussian_loss_type: mse
  weight_decay: 1.0e-05
  scheduler: cosine
  data_split_ratios:
  - 0.99
  - 0.005
  - 0.005
  merge_categoricals_into_numerical: false



## Load the Table
IMPORTANT: code expects `{table}_domain.json` and `dataset_meta.json` files under `base_data_dir`

In [9]:
log(INFO, f"Loading data from {base_data_dir}...")
tables, relation_order, _ = load_tables(Path(base_data_dir))
log(INFO, f"relation order is {relation_order}")

INFO :      Loading data from /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/single_table/data...
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (16000, 8)
INFO :      Total dataframe shape: (16000, 8)
INFO :      Numerical data shape: (16000, 4)
INFO :      Categorical data shape: (16000, 4)
INFO :      relation order is [(None, 'trans')]


## Train the model

In [10]:
log(INFO, "Training model...")
diffusion_config = ClavaDDPMDiffusionConfig(**cfg.diffusion_config)
tables, _ = clava_training(
    tables,
    relation_order,
    Path(base_output_dir),
    diffusion_config,
    device=DEVICE,
)
log(INFO, "Model trained successfully.")

INFO :      Training model...
INFO :      Training None -> trans model from scratch
INFO :      No cache_dir provided. Will not attempt to load or save transformed dataset from/to cache
INFO :      No NaN processing policy specified.
INFO :      Model params: ModelParameters(diffusion_parameters=DiffusionParameters(layers_dimensions=[512, 1024, 1024, 1024, 1024, 512], dropout=0.0, input_dimension=0, output_dimension=0, embedding_dimension=0, n_blocks=0, block_dimension=0, hidden_dimension=0, dropout_first=0, dropout_second=0), input_dimension=np.int64(36), num_classes=0, is_target_conditioned=<IsTargetConditioned.NONE: 'none'>)
INFO :      Getting model: mlp
INFO :      Step 500/20000 MLoss: 0.5847 GLoss: 0.9315 Sum: 1.5162
INFO :      Step 1000/20000 MLoss: 0.5105 GLoss: 0.6391 Sum: 1.1496
INFO :      Step 1500/20000 MLoss: 0.4951 GLoss: 0.4284 Sum: 0.9235


KeyboardInterrupt: 

## Save Results

In [8]:
results_file = Path(base_output_dir) / "models" / "None_trans_ckpt.pkl"
log(INFO, f"Checking the results from {results_file}...")

with open(results_file, "rb") as f:
    result = pickle.load(f)

# Asserting the results are the correct type
assert isinstance(result, ClavaDDPMModelArtifacts)

log(INFO, f"Result size (in bytes): {results_file.stat().st_size}")

INFO :      Checking the results from /home/coder/synthetic-data-bootcamp/implementations/tabular_data/single_table/results/models/None_trans_ckpt.pkl...
INFO :      Result size (in bytes): 18927312
